In [16]:
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

model_name = "gpt-oss:20b"
chat_generator = OllamaChatGenerator(model=model_name,
                            url = "http://141.2.11.133:11434",
                            timeout = 30*60,
                            generation_kwargs={
                              "temperature": 0,
                              "num_ctx": 8192
                              })

In [17]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()

In [18]:
from haystack import Document
from pypdf import PdfReader
import os

pdf_folder = r"C:\Users\olena\Olena_Tsvietkova\Forschungsmethoden\dat_project\notebooks\downloads"

def load_pdfs(folder):
    docs = []

    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            path = os.path.join(folder, file)
            reader = PdfReader(path)

            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    docs.append(
                        Document(
                            content=text,
                            meta={"source": file, "page": i}
                        )
                    )
    return docs


docs = load_pdfs(pdf_folder)
print(len(docs))

78


In [19]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()

In [20]:
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage

prompt_template = [
    ChatMessage.from_system("""
You extract structured data from scientific text.

TASK:
Return a JSON object with EXACTLY these three fields:
1. title_and_authors → paper title + authors in one string
2. bibtex → valid BibTeX entry
3. summary → short academic summary

RULES:
- Output ONLY valid JSON
- No explanations
- No markdown
- No extra text

TEXT:
{{ passage }}

SCHEMA:
{{ schema }}
""")
]

prompt_builder = ChatPromptBuilder(template=prompt_template)

ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [21]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
doc_embedder.warm_up()

result = doc_embedder.run(docs)
embedded_docs = result["documents"]

Batches: 100%|██████████| 3/3 [00:01<00:00,  1.91it/s]


In [22]:
document_store.write_documents(embedded_docs)

78

In [23]:
from haystack.components.retrievers import InMemoryEmbeddingRetriever

retriever = InMemoryEmbeddingRetriever(document_store=document_store, top_k=3)

In [24]:
from haystack import Pipeline

pipeline = Pipeline(max_runs_per_component=5)

pipeline.add_component("prompt_builder", prompt_builder)
pipeline.add_component("llm", chat_generator)

pipeline.connect("prompt_builder.prompt", "llm.messages")


🚅 Components
  - prompt_builder: ChatPromptBuilder
  - llm: OllamaChatGenerator
🛤️ Connections
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

In [25]:
passage = "\n\n".join([d.content for d in docs[:5]])

result = pipeline.run({
    "prompt_builder": {
        "passage": passage,
   
    }
})

print(result)

{'llm': {'replies': [ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[ReasoningContent(reasoning_text='We need to produce JSON with fields: title_and_authors, bibtex, summary. The text is a paper titled "Agentic AI Needs a Systems Theory" with authors list. We need to produce a valid BibTeX entry. We need to parse authors: Erik Miehling, Karthikeyan Natesan, Ramamurthy Kush R. Varshney, Matthew Riemer, Djallel Bouneffouf, John T. Richards, Amit Dhurandhar, Elizabeth M. Daly, Michael Hind, Prasanna Sattigeri, Dennis Wei, Ambrish Rawat, Jasmina Gajcin, Werner Geyer. Actually list: "Erik Miehling Karthikeyan Natesan Ramamurthy Kush R. Varshney Matthew Riemer Djallel Bouneffouf John T. Richards Amit Dhurandhar Elizabeth M. Daly Michael Hind Prasanna Sattigeri Dennis Wei Ambrish Rawat Jasmina Gajcin Werner Geyer". That\'s 14 authors. We need to format BibTeX accordingly. Title: "Agentic AI Needs a Systems Theory". Journal? It\'s a position paper, maybe from arXiv. The arXiv ID

In [26]:
output = result["llm"]["replies"][0].text
print(output)

{"title_and_authors":"Agentic AI Needs a Systems Theory – Erik Miehling, Karthikeyan Natesan, Ramamurthy Kush R. Varshney, Matthew Riemer, Djallel Bouneffouf, John T. Richards, Amit Dhurandhar, Elizabeth M. Daly, Michael Hind, Prasanna Sattigeri, Dennis Wei, Ambrish Rawat, Jasmina Gajcin, Werner Geyer","bibtex":"@article{miehling2025agentic,\\n  author = {Erik Miehling and Karthikeyan Natesan and Ramamurthy Kush R. Varshney and Matthew Riemer and Djallel Bouneffouf and John T. Richards and Amit Dhurandhar and Elizabeth M. Daly and Michael Hind and Prasanna Sattigeri and Dennis Wei and Ambrish Rawat and Jasmina Gajcin and Werner Geyer},\\n  title = {Agentic AI Needs a Systems Theory},\\n  year = {2025},\\n  eprint = {2503.00237},\\n  archiveprefix = {arXiv},\\n  primaryclass = {cs.AI},\\n  url = {https://arxiv.org/abs/2503.00237},\\n  publisher = {IBM Research},\\n  note = {Position paper}\\n}","summary":"This position paper argues that the development of agentic AI—autonomous agents ca